# Submissions por Algoritmo

Este notebook genera un `submission_<modelo>.csv` por cada algoritmo de recomendación.
Todas las predicciones sobre el test están **vectorizadas** (NumPy matricial, sin loops Python)
para minimizar el tiempo de inferencia.

| Archivo generado | Algoritmo |
|---|---|
| `submission_media_global.csv` | Media global |
| `submission_media_item.csv` | Media por ítem |
| `submission_media_usuario.csv` | Media por usuario |
| `submission_media_shrunk.csv` | Media shrinkada bayesiana |
| `submission_knn_user.csv` | KNN user-based (Surprise) |
| `submission_knn_item.csv` | KNN item-based (Surprise) |
| `submission_pmf.csv` | PMF / Simon Funk SVD (Numba) |
| `submission_nmf.csv` | NMF (sklearn) |
| `submission_bnmf.csv` | BNMF proxy (NMF regularizado) |

In [2]:
# ══════════════════════════════════════════════════════════════════════════
# 0. SETUP — imports, semillas, rutas
# ══════════════════════════════════════════════════════════════════════════
from IPython import get_ipython
import time

ip = get_ipython()
try:
    for cb in list(ip.events.callbacks['pre_run_cell']):
        ip.events.unregister('pre_run_cell', cb)
    for cb in list(ip.events.callbacks['post_run_cell']):
        ip.events.unregister('post_run_cell', cb)
    print("Callbacks limpiados")
except:
    print("Nada que limpiar")

try:
    from plyer import notification
    class ExecutionTimer:
        def __init__(self, threshold=60):
            self.start_time = None
            self.threshold  = threshold
        def pre_run(self, info):
            self.start_time = time.time()
        def post_run(self, result):
            if self.start_time:
                duration = time.time() - self.start_time
                if duration > self.threshold:
                    notification.notify(
                        title="¡Celda Finalizada!",
                        message=f"La ejecución tomó {duration:.2f}s.",
                        app_name="Jupyter", timeout=10
                    )
                self.start_time = None
    timer = ExecutionTimer(threshold=60)
    ip.events.register('pre_run_cell',  timer.pre_run)
    ip.events.register('post_run_cell', timer.post_run)
    print("Notificador configurado para celdas > 60s")
except ImportError:
    print("plyer no disponible — sin notificaciones")

Callbacks limpiados
Notificador configurado para celdas > 60s


In [3]:
import random, json, os
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from numba import njit

# ── Rutas ─────────────────────────────────────────────────────────────────
TRAIN_PATH    = 'train.csv'
TEST_PATH     = 'test.csv'
GRID_LOG      = 'grid_search_log.csv'
BEST_CFG_PATH = 'best_cfg.json'
MODEL_PATH    = 'pmf_final.npz'
SUBMISSIONS_DIR = '.'   # carpeta donde se escriben los CSVs

RATING_MIN = 1.0
RATING_MAX = 10.0
SEED       = 42

random.seed(SEED)
np.random.seed(SEED)
print("Imports OK")

Imports OK


In [4]:
# ══════════════════════════════════════════════════════════════════════════
# 1. CARGA DE DATOS Y ENCODERS
# ══════════════════════════════════════════════════════════════════════════
df_ratings = pd.read_csv(TRAIN_PATH)[['user', 'item', 'rating']]
print(f'Train raw: {df_ratings.shape} | ratings [{df_ratings.rating.min()}, {df_ratings.rating.max()}]')

# ── Encoders ──────────────────────────────────────────────────────────────
user2idx = {u: i for i, u in enumerate(sorted(df_ratings['user'].unique()))}
item2idx = {it: i for i, it in enumerate(sorted(df_ratings['item'].unique()))}
idx2user = {i: u for u, i in user2idx.items()}
idx2item = {i: it for it, i in item2idx.items()}

df = df_ratings.copy()
df['user'] = df['user'].map(user2idx)
df['item'] = df['item'].map(item2idx)

N_USERS = len(user2idx)
N_ITEMS = len(item2idx)

# ── Split random 80/20 ────────────────────────────────────────────────────
np.random.seed(SEED)
idx_all  = np.random.permutation(len(df))
n_val    = int(len(df) * 0.20)
val_df   = df.iloc[idx_all[:n_val]].reset_index(drop=True)
train_df = df.iloc[idx_all[n_val:]].reset_index(drop=True)

# ── Split cold-start (val = usuarios distintos a train) ───────────────────
np.random.seed(SEED)
all_users     = df['user'].unique()
val_users     = set(np.random.choice(all_users, size=int(len(all_users) * 0.20), replace=False))
val_df_cold   = df[df['user'].isin(val_users)].reset_index(drop=True)
train_df_cold = df[~df['user'].isin(val_users)].reset_index(drop=True)

# ── Test competición ──────────────────────────────────────────────────────
df_test_comp = pd.read_csv(TEST_PATH)
df_test_comp['user'] = df_test_comp['user'].map(user2idx).fillna(-1).astype(int)
df_test_comp['item'] = df_test_comp['item'].map(item2idx).fillna(-1).astype(int)

# Arrays numpy del test — se usan en todas las predicciones vectorizadas
TEST_USERS = df_test_comp['user'].values   # int array
TEST_ITEMS = df_test_comp['item'].values   # int array
TEST_IDS   = df_test_comp['ID'].values

# ── Matriz sparse (train) ─────────────────────────────────────────────────
R_sparse = csr_matrix(
    (train_df['rating'].values, (train_df['user'].values, train_df['item'].values)),
    shape=(N_USERS, N_ITEMS)
)

print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')
print(f'Train cold: {len(train_df_cold):,} | Val cold: {len(val_df_cold):,}')
print(f'N_USERS={N_USERS} | N_ITEMS={N_ITEMS}')
print(f'Test competición: {len(df_test_comp):,} pares')

Train raw: (390351, 3) | ratings [1.0, 10.0]
Train: 312,281 | Val: 78,070
Train cold: 313,620 | Val cold: 76,731
N_USERS=73456 | N_ITEMS=171171
Test competición: 43,320 pares


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# 2. HELPERS: make_submission, mae, evaluar (vectorizado)
# ══════════════════════════════════════════════════════════════════════════

resultados = []   # acumula {'modelo': ..., 'MAE': ...}


def make_submission(nombre_modelo: str, preds: np.ndarray) -> str:
    """
    Recibe el array de predicciones (ya en orden de TEST_IDS),
    aplica clip [RATING_MIN, RATING_MAX], redondea a 4 decimales
    y escribe submission_<nombre_modelo>.csv.
    Devuelve la ruta del archivo generado.
    """
    preds_clipped = np.clip(preds, RATING_MIN, RATING_MAX).round(4)
    fname = os.path.join(SUBMISSIONS_DIR, f'submission_{nombre_modelo}.csv')
    pd.DataFrame({'ID': TEST_IDS, 'rating': preds_clipped}).to_csv(fname, index=False)
    print(f'  ✓ {fname}  ({len(preds_clipped):,} filas, '
          f'rating ∈ [{preds_clipped.min():.2f}, {preds_clipped.max():.2f}])')
    return fname


def mae(y_true, y_pred):
    return float(np.mean(np.abs(np.asarray(y_true, float) - np.asarray(y_pred, float))))


def evaluar_vectorizado(nombre: str, preds_val: np.ndarray, val_df_local=None) -> dict:
    """
    Calcula MAE sobre val_df dado un array de predicciones val.
    preds_val debe estar alineado con val_df_local (o val_df global).
    """
    if val_df_local is None:
        val_df_local = val_df
    y_true = val_df_local['rating'].values
    score  = mae(y_true, np.clip(preds_val, RATING_MIN, RATING_MAX))
    print(f'  {nombre:<25}  VAL MAE = {score:.4f}')
    return {'modelo': nombre, 'MAE': score}


print("Helpers definidos OK")

Helpers definidos OK


In [6]:
# ══════════════════════════════════════════════════════════════════════════
# 3. CLASE PMF (Numba SGD + predict_batch vectorizado)
# ══════════════════════════════════════════════════════════════════════════
@njit
def _sgd_epoch(users, items, ratings, U, V, bu, bi, mu, lr, reg, reg_bias):
    idx = np.random.permutation(len(users))
    total_err = 0.0
    for i in idx:
        u, v, r = users[i], items[i], ratings[i]
        pred = mu + bu[u] + bi[v] + np.dot(U[u], V[v])
        err  = r - pred
        total_err += abs(err)
        bu[u] += lr * (err - reg_bias * bu[u])
        bi[v] += lr * (err - reg_bias * bi[v])
        U[u]  += lr * (err * V[v] - reg * U[u])
        V[v]  += lr * (err * U[u] - reg * V[v])
    return total_err / len(users)


class PMF:
    """
    PMF con bias (Simon Funk SVD, Koren 2008).
    SGD compilado con Numba + Bayesian shrinkage en cold-start.
    predict_batch: predicción vectorizada sobre arrays numpy.
    """

    def __init__(self, n_factors=50, n_epochs=100, lr=0.005,
                 reg=0.02, reg_bias=0.005, patience=5, shrink_k=10):
        self.n_factors = n_factors
        self.n_epochs  = n_epochs
        self.lr        = lr
        self.reg       = reg
        self.reg_bias  = reg_bias
        self.patience  = patience
        self.shrink_k  = shrink_k
        self.train_mae = []
        self.val_mae   = []

    # ── Entrenamiento ──────────────────────────────────────────────────────
    def fit(self, train_df_local, val_df_local=None, seed=SEED):
        np.random.seed(seed)

        full_df = train_df_local if val_df_local is None else pd.concat([train_df_local, val_df_local])
        n_u = full_df['user'].max() + 1
        n_i = full_df['item'].max() + 1

        self.U  = np.random.normal(0, 0.01, (n_u, self.n_factors))
        self.V  = np.random.normal(0, 0.01, (n_i, self.n_factors))
        self.bu = np.zeros(n_u)
        self.bi = np.zeros(n_i)
        self.mu = train_df_local['rating'].mean()

        # Fallback vectores (Bayesian shrinkage)
        k = self.shrink_k
        item_stats = train_df_local.groupby('item')['rating'].agg(['mean', 'count'])
        self.item_mean = {
            item: (row['count'] * row['mean'] + k * self.mu) / (row['count'] + k)
            for item, row in item_stats.iterrows()
        }
        user_stats = train_df_local.groupby('user')['rating'].agg(['mean', 'count'])
        self.user_mean = {
            user: (row['count'] * row['mean'] + k * self.mu) / (row['count'] + k)
            for user, row in user_stats.iterrows()
        }
        # arrays numpy de fallback (para predict_batch)
        self._user_mean_arr = np.full(n_u, self.mu)
        for uid, val in self.user_mean.items():
            if uid < n_u:
                self._user_mean_arr[uid] = val
        self._item_mean_arr = np.full(n_i, self.mu)
        for iid, val in self.item_mean.items():
            if iid < n_i:
                self._item_mean_arr[iid] = val

        users_arr   = train_df_local['user'].values.astype(np.int64)
        items_arr   = train_df_local['item'].values.astype(np.int64)
        ratings_arr = train_df_local['rating'].values.astype(np.float64)

        best_val          = float('inf')
        epochs_no_improve = 0
        best_state        = None

        print('  Compilando Numba (solo primera vez)...')
        for epoch in range(self.n_epochs):
            t_mae = _sgd_epoch(
                users_arr, items_arr, ratings_arr,
                self.U, self.V, self.bu, self.bi, self.mu,
                self.lr, self.reg, self.reg_bias
            )
            self.train_mae.append(t_mae)

            if val_df_local is not None:
                v_preds = self.predict_batch(
                    val_df_local['user'].values, val_df_local['item'].values
                )
                v_mae = mae(val_df_local['rating'].values, v_preds)
                self.val_mae.append(v_mae)

                if v_mae < best_val:
                    best_val          = v_mae
                    epochs_no_improve = 0
                    best_state = {k: getattr(self, k).copy() for k in ['U', 'V', 'bu', 'bi']}
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= self.patience:
                        print(f'  Early stopping en época {epoch+1} | mejor val MAE: {best_val:.4f}')
                        for k, v in best_state.items():
                            setattr(self, k, v)
                        break

            if (epoch + 1) % 10 == 0:
                msg = f'  Epoch {epoch+1:>3}/{self.n_epochs} | train: {self.train_mae[-1]:.4f}'
                if val_df_local is not None:
                    msg += f' | val: {self.val_mae[-1]:.4f}'
                print(msg)
        return self

    # ── Predicción vectorizada ─────────────────────────────────────────────
    def predict_batch(self, users: np.ndarray, items: np.ndarray) -> np.ndarray:
        """
        Predicción vectorizada: recibe arrays de usuarios e items (int),
        devuelve array float sin clip (el clip lo aplica make_submission).
        Gestiona cold-start por máscara booleana.
        """
        n_u, n_i = self.U.shape[0], self.V.shape[0]
        u_known  = (users >= 0) & (users < n_u)
        i_known  = (items >= 0) & (items < n_i)

        preds = np.full(len(users), self.mu, dtype=np.float64)

        # ambos conocidos → dot product completo
        mask_both = u_known & i_known
        if mask_both.any():
            uu = users[mask_both]
            ii = items[mask_both]
            preds[mask_both] = (
                self.mu
                + self.bu[uu]
                + self.bi[ii]
                + (self.U[uu] * self.V[ii]).sum(axis=1)
            )

        # solo usuario cold-start → media shrunk del item
        mask_ucold = (~u_known) & i_known
        if mask_ucold.any():
            preds[mask_ucold] = self._item_mean_arr[items[mask_ucold]]

        # solo item cold-start → media shrunk del usuario
        mask_icold = u_known & (~i_known)
        if mask_icold.any():
            preds[mask_icold] = self._user_mean_arr[users[mask_icold]]

        # ambos cold → mu global (ya inicializado)
        return preds

    # ── Predicción escalar (compatibilidad) ────────────────────────────────
    def predict(self, user, item):
        return self.predict_batch(np.array([user]), np.array([item]))[0]

    # ── Persistencia ──────────────────────────────────────────────────────
    def save(self, path=MODEL_PATH):
        np.savez_compressed(
            path,
            U=self.U, V=self.V, bu=self.bu, bi=self.bi,
            mu=np.array([self.mu]),
            shrink_k=np.array([self.shrink_k]),
            train_mae=np.array(self.train_mae),
            val_mae=np.array(self.val_mae),
            user_mean_keys=np.array(list(self.user_mean.keys())),
            user_mean_vals=np.array(list(self.user_mean.values())),
            item_mean_keys=np.array(list(self.item_mean.keys())),
            item_mean_vals=np.array(list(self.item_mean.values())),
        )
        print(f'  Modelo guardado → {path}')

    @classmethod
    def load(cls, path=MODEL_PATH):
        data = np.load(path, allow_pickle=False)
        m = cls.__new__(cls)
        m.U         = data['U']
        m.V         = data['V']
        m.bu        = data['bu']
        m.bi        = data['bi']
        m.mu        = float(data['mu'][0])
        m.shrink_k  = int(data['shrink_k'][0])
        m.train_mae = list(data['train_mae'])
        m.val_mae   = list(data['val_mae'])
        m.user_mean = dict(zip(data['user_mean_keys'].tolist(),
                               data['user_mean_vals'].tolist()))
        m.item_mean = dict(zip(data['item_mean_keys'].tolist(),
                               data['item_mean_vals'].tolist()))
        m.n_factors = m.U.shape[1]
        n_u, n_i = m.U.shape[0], m.V.shape[0]
        m._user_mean_arr = np.full(n_u, m.mu)
        for uid, val in m.user_mean.items():
            if uid < n_u:
                m._user_mean_arr[uid] = val
        m._item_mean_arr = np.full(n_i, m.mu)
        for iid, val in m.item_mean.items():
            if iid < n_i:
                m._item_mean_arr[iid] = val
        print(f'  Modelo cargado ← {path}  (U={m.U.shape}, shrink_k={m.shrink_k})')
        return m


print("Clase PMF definida OK")

Clase PMF definida OK


In [7]:
# ══════════════════════════════════════════════════════════════════════════
# 4. BASELINES DE MEDIA  (predicción vectorizada con pd.Series.map)
# ══════════════════════════════════════════════════════════════════════════
mu_global_train  = train_df['rating'].mean()
item_means_train = train_df.groupby('item')['rating'].mean()
user_means_train = train_df.groupby('user')['rating'].mean()

# ──── Media global ────────────────────────────────────────────────────────
preds_test_mg  = np.full(len(TEST_IDS), mu_global_train)
preds_val_mg   = np.full(len(val_df), mu_global_train)
res = evaluar_vectorizado('MediaGlobal', preds_val_mg)
resultados.append(res)
make_submission('media_global', preds_test_mg)

# ──── Media por ítem ──────────────────────────────────────────────────────
preds_test_mi  = (
    pd.Series(TEST_ITEMS)
    .map(item_means_train)
    .fillna(mu_global_train)
    .values
)
preds_val_mi   = (
    val_df['item']
    .map(item_means_train)
    .fillna(mu_global_train)
    .values
)
res = evaluar_vectorizado('MediaItem', preds_val_mi)
resultados.append(res)
make_submission('media_item', preds_test_mi)

# ──── Media por usuario ───────────────────────────────────────────────────
preds_test_mu  = (
    pd.Series(TEST_USERS)
    .map(user_means_train)
    .fillna(mu_global_train)
    .values
)
preds_val_mu   = (
    val_df['user']
    .map(user_means_train)
    .fillna(mu_global_train)
    .values
)
res = evaluar_vectorizado('MediaUsuario', preds_val_mu)
resultados.append(res)
make_submission('media_usuario', preds_test_mu)

# ──── Media shrinkada (Bayesian) ──────────────────────────────────────────
K_SHRINK = 10
item_stats  = train_df.groupby('item')['rating'].agg(['mean', 'count'])
item_shrunk = (
    (item_stats['count'] * item_stats['mean'] + K_SHRINK * mu_global_train)
    / (item_stats['count'] + K_SHRINK)
)
user_stats  = train_df.groupby('user')['rating'].agg(['mean', 'count'])
user_shrunk = (
    (user_stats['count'] * user_stats['mean'] + K_SHRINK * mu_global_train)
    / (user_stats['count'] + K_SHRINK)
)

def _shrunk_batch(users_arr, items_arr):
    u_s = pd.Series(users_arr).map(user_shrunk)
    i_s = pd.Series(items_arr).map(item_shrunk)
    u_na = u_s.isna().values
    i_na = i_s.isna().values
    u_s  = u_s.fillna(mu_global_train).values
    i_s  = i_s.fillna(mu_global_train).values
    preds = np.where(u_na & i_na, mu_global_train,
            np.where(u_na, i_s,
            np.where(i_na, u_s,
                     (u_s + i_s) / 2.0)))
    return preds

preds_test_sh = _shrunk_batch(TEST_USERS, TEST_ITEMS)
preds_val_sh  = _shrunk_batch(val_df['user'].values, val_df['item'].values)
res = evaluar_vectorizado(f'MediaShrunk(k={K_SHRINK})', preds_val_sh)
resultados.append(res)
make_submission('media_shrunk', preds_test_sh)

  MediaGlobal                VAL MAE = 1.4966
  ✓ ./submission_media_global.csv  (43,320 filas, rating ∈ [7.60, 7.60])
  MediaItem                  VAL MAE = 1.5369
  ✓ ./submission_media_item.csv  (43,320 filas, rating ∈ [1.00, 10.00])
  MediaUsuario               VAL MAE = 1.2881
  ✓ ./submission_media_usuario.csv  (43,320 filas, rating ∈ [1.00, 10.00])
  MediaShrunk(k=10)          VAL MAE = 1.3178
  ✓ ./submission_media_shrunk.csv  (43,320 filas, rating ∈ [4.29, 9.45])


'./submission_media_shrunk.csv'

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 5. KNN USER-BASED  (Surprise — .test() batch nativo)
# ══════════════════════════════════════════════════════════════════════════
from surprise import KNNWithMeans, Dataset, Reader
from surprise import accuracy as surp_acc

reader     = Reader(rating_scale=(RATING_MIN, RATING_MAX))
train_data = Dataset.load_from_df(train_df[['user', 'item', 'rating']], reader)
trainset   = train_data.build_full_trainset()

# Testsets de val y competición (formato Surprise)
testset_val  = [(row.user, row.item, row.rating) for row in val_df.itertuples()]
testset_comp = [(int(u), int(i), 0.0) for u, i in zip(TEST_USERS, TEST_ITEMS)]

print('Entrenando KNN user-based...')
knn_user = KNNWithMeans(
    k=40, min_k=2,
    sim_options={'name': 'pearson_baseline', 'user_based': True},
    verbose=False
)
knn_user.fit(trainset)

# Evaluación sobre val
preds_val_ku  = knn_user.test(testset_val)
mae_knn_user  = surp_acc.mae(preds_val_ku, verbose=False)
print(f'  KNN_user               VAL MAE = {mae_knn_user:.4f}')
resultados.append({'modelo': 'KNN_user', 'MAE': mae_knn_user})

# Submission (batch .test() sobre test de competición)
preds_comp_ku = np.array([p.est for p in knn_user.test(testset_comp)])
make_submission('knn_user', preds_comp_ku)

Entrenando KNN user-based...
  KNN_user               VAL MAE = 1.3832
  ✓ ./submission_knn_user.csv  (43,320 filas, rating ∈ [1.00, 10.00])


'./submission_knn_user.csv'

: 

In [10]:
# ══════════════════════════════════════════════════════════════════════════
# 6. KNN ITEM-BASED  (Surprise — .test() batch nativo)
# ══════════════════════════════════════════════════════════════════════════
from surprise import KNNWithMeans, Dataset, Reader
from surprise import accuracy as surp_acc

reader     = Reader(rating_scale=(RATING_MIN, RATING_MAX))
train_data = Dataset.load_from_df(train_df[['user', 'item', 'rating']], reader)
trainset   = train_data.build_full_trainset()

print('Entrenando KNN item-based...')
knn_item = KNNWithMeans(
    k=40, min_k=2,
    sim_options={'name': 'pearson_baseline', 'user_based': False},
    verbose=False
)
knn_item.fit(trainset)

# Evaluación sobre val
preds_val_ki  = knn_item.test(testset_val)
mae_knn_item  = surp_acc.mae(preds_val_ki, verbose=False)
print(f'  KNN_item               VAL MAE = {mae_knn_item:.4f}')
resultados.append({'modelo': 'KNN_item', 'MAE': mae_knn_item})

# Submission
preds_comp_ki = np.array([p.est for p in knn_item.test(testset_comp)])
make_submission('knn_item', preds_comp_ki)

Entrenando KNN item-based...


: 

In [8]:
# ══════════════════════════════════════════════════════════════════════════
# 7. PMF  (Numba SGD con bias — predict_batch vectorizado)
# Carga desde disco si existe para evitar reentrenamiento.
# ══════════════════════════════════════════════════════════════════════════
if os.path.exists(MODEL_PATH):
    print(f'Modelo encontrado en {MODEL_PATH} — cargando sin reentrenar...')
    pmf = PMF.load(MODEL_PATH)
else:
    print('Entrenando PMF...')
    random.seed(SEED); np.random.seed(SEED)

    # Si existe best_cfg.json, lo usamos; si no, parámetros por defecto
    if os.path.exists(BEST_CFG_PATH):
        with open(BEST_CFG_PATH) as f:
            ckpt = json.load(f)
        best_cfg     = {k: v for k, v in ckpt.items() if k not in ('val_mae', 'n_epochs_opt')}
        n_epochs_opt = ckpt.get('n_epochs_opt', 100)
        print(f'  Config desde {BEST_CFG_PATH}: {best_cfg} | epochs={n_epochs_opt}')
    else:
        best_cfg     = {'n_factors': 200, 'lr': 0.01, 'reg': 0.02, 'reg_bias': 0.001, 'shrink_k': 10}
        n_epochs_opt = 100
        print(f'  Usando config por defecto: {best_cfg}')

    # Entrenar sobre train+val completo (sin val para early stopping)
    full_train = pd.concat([train_df, val_df]).reset_index(drop=True)
    pmf = PMF(
        n_factors = best_cfg['n_factors'],
        n_epochs  = n_epochs_opt,
        lr        = best_cfg['lr'],
        reg       = best_cfg['reg'],
        reg_bias  = best_cfg['reg_bias'],
        patience  = 999,
        shrink_k  = best_cfg.get('shrink_k', 10)
    )
    pmf.fit(full_train)
    pmf.save(MODEL_PATH)

# ── Evaluación (sobre val_df) ──────────────────────────────────────────────
preds_val_pmf  = pmf.predict_batch(val_df['user'].values, val_df['item'].values)
res = evaluar_vectorizado('PMF', preds_val_pmf)
resultados.append(res)

# ── Submission ────────────────────────────────────────────────────────────
preds_test_pmf = pmf.predict_batch(TEST_USERS, TEST_ITEMS)
make_submission('pmf', preds_test_pmf)

Entrenando PMF...
  Usando config por defecto: {'n_factors': 200, 'lr': 0.01, 'reg': 0.02, 'reg_bias': 0.001, 'shrink_k': 10}
  Compilando Numba (solo primera vez)...
  Epoch  10/100 | train: 1.0847
  Epoch  20/100 | train: 0.8347
  Epoch  30/100 | train: 0.5741
  Epoch  40/100 | train: 0.3962
  Epoch  50/100 | train: 0.2838
  Epoch  60/100 | train: 0.2117
  Epoch  70/100 | train: 0.1637
  Epoch  80/100 | train: 0.1309
  Epoch  90/100 | train: 0.1076
  Epoch 100/100 | train: 0.0908
  Modelo guardado → pmf_final.npz
  PMF                        VAL MAE = 0.0870
  ✓ ./submission_pmf.csv  (43,320 filas, rating ∈ [1.53, 10.00])


'./submission_pmf.csv'

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 8. NMF  (sklearn — predicción matricial W[users] * H.T[items])
# ══════════════════════════════════════════════════════════════════════════
from sklearn.decomposition import NMF as SklearnNMF
from scipy.sparse import csr_matrix as csr

print('Entrenando NMF (sklearn)...')

R_train_sp = csr(
    (train_df['rating'].values,
     (train_df['user'].values, train_df['item'].values)),
    shape=(N_USERS, N_ITEMS),
    dtype=np.float32
)

NMF_FACTORS = 200
nmf_model = SklearnNMF(
    n_components  = NMF_FACTORS,
    init          = 'nndsvda',
    max_iter      = 300,
    random_state  = SEED,
    l1_ratio      = 0.0,
    alpha_W       = 0.01,
    alpha_H       = 'same',
    verbose       = 0
)
W_nmf = nmf_model.fit_transform(R_train_sp)   # (N_USERS, k)
H_nmf = nmf_model.components_                 # (k, N_ITEMS)
mu_nmf = train_df['rating'].mean()

def _nmf_batch(users_arr, items_arr):
    """
    Predicción vectorizada NMF.
    Cold-start (u<0 o i<0) → mu_nmf.
    """
    preds = np.full(len(users_arr), mu_nmf, dtype=np.float64)
    mask  = ((users_arr >= 0) & (users_arr < N_USERS) &
             (items_arr >= 0) & (items_arr < N_ITEMS))
    if mask.any():
        uu = users_arr[mask]
        ii = items_arr[mask]
        preds[mask] = (W_nmf[uu] * H_nmf[:, ii].T).sum(axis=1)
    return preds

# ── Evaluación ────────────────────────────────────────────────────────────
preds_val_nmf  = _nmf_batch(val_df['user'].values, val_df['item'].values)
res = evaluar_vectorizado('NMF', preds_val_nmf)
resultados.append(res)

# ── Submission ────────────────────────────────────────────────────────────
preds_test_nmf = _nmf_batch(TEST_USERS, TEST_ITEMS)
make_submission('nmf', preds_test_nmf)


Entrenando NMF (sklearn)...
  NMF                        VAL MAE = 6.6162
  ✓ ./submission_nmf.csv  (43,320 filas, rating ∈ [1.00, 7.60])


'./submission_nmf.csv'

: 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 9. BNMF  (nimfa si disponible, proxy NMF regularizado si no)
# ══════════════════════════════════════════════════════════════════════════
try:
    import nimfa
    NIMFA_OK = True
except ImportError:
    NIMFA_OK = False
    print('nimfa no instalado — usando NMF L1+L2 regularizado como proxy BNMF')

if NIMFA_OK:
    print('Entrenando BNMF (nimfa)...')
    R_dense = R_train_sp.toarray().astype(np.float64)
    bnmf = nimfa.Bmf(
        R_dense, rank=50, max_iter=100,
        lambda_w=1.2, lambda_h=1.2
    )
    bnmf_fit = bnmf()
    W_b = np.array(bnmf_fit.basis())
    H_b = np.array(bnmf_fit.coef())
    modelo_bnmf = 'BNMF_nimfa'
else:
    print('Entrenando BNMF-proxy (NMF L1+L2 regularizado)...')
    bnmf_proxy = SklearnNMF(
        n_components  = 100,
        init          = 'nndsvda',
        max_iter      = 300,
        random_state  = SEED,
        l1_ratio      = 0.5,
        alpha_W       = 0.1,
        alpha_H       = 'same',
        verbose       = 0
    )
    W_b = bnmf_proxy.fit_transform(R_train_sp)
    H_b = bnmf_proxy.components_
    modelo_bnmf = 'BNMF_proxy'

def _bnmf_batch(users_arr, items_arr):
    """Predicción vectorizada BNMF (misma estructura que NMF)."""
    preds = np.full(len(users_arr), mu_nmf, dtype=np.float64)
    mask  = ((users_arr >= 0) & (users_arr < W_b.shape[0]) &
             (items_arr >= 0) & (items_arr < H_b.shape[1]))
    if mask.any():
        uu = users_arr[mask]
        ii = items_arr[mask]
        preds[mask] = (W_b[uu] * H_b[:, ii].T).sum(axis=1)
    return preds

# ── Evaluación ────────────────────────────────────────────────────────────
preds_val_bnmf  = _bnmf_batch(val_df['user'].values, val_df['item'].values)
res = evaluar_vectorizado(modelo_bnmf, preds_val_bnmf)
resultados.append(res)

# ── Submission ────────────────────────────────────────────────────────────
preds_test_bnmf = _bnmf_batch(TEST_USERS, TEST_ITEMS)
make_submission('bnmf', preds_test_bnmf)


/Users/kzzazzk/Documents/MAADM/2C/2ª 5S/RECSYS/tecnicas-colaborativas/.venv/lib/python3.12/site-packages/nimfa/models/nmf.py:589: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if self.seed is not None and self.seed is not "fixed":
/Users/kzzazzk/Documents/MAADM/2C/2ª 5S/RECSYS/tecnicas-colaborativas/.venv/lib/python3.12/site-packages/nimfa/methods/seeding/random.py:60: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  if sn[0] is 'S' and sn[1:].isdigit():
/Users/kzzazzk/Documents/MAADM/2C/2ª 5S/RECSYS/tecnicas-colaborativas/.venv/lib/python3.12/site-packages/nimfa/models/smf.py:114: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if self.seed is not None and self.seed is not "fixed":
/Users/kzzazzk/Documents/MAADM/2C/2ª 5S/RECSYS/tecnicas-colaborativas/.venv/lib/python3.12/site-packages/nimfa/methods/factorization/sepnmf.py:276: SyntaxWarning: invalid escape sequence '\m'
  .. math:: \arg\min_{Y \ge 0} \| V - W H \|_F.


Entrenando BNMF (nimfa)...


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 10. RANKING COMPARATIVO + RESUMEN DE SUBMISSIONS
# ══════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

df_res = pd.DataFrame(resultados).sort_values('MAE').reset_index(drop=True)
df_res.index += 1
df_res['Δ vs mejor'] = (df_res['MAE'] - df_res['MAE'].iloc[0]).round(4)

# Añadir columna con nombre del archivo de submission
nombre_to_csv = {
    'MediaGlobal':          'submission_media_global.csv',
    'MediaItem':            'submission_media_item.csv',
    'MediaUsuario':         'submission_media_usuario.csv',
    f'MediaShrunk(k={K_SHRINK})': 'submission_media_shrunk.csv',
    'KNN_user':             'submission_knn_user.csv',
    'KNN_item':             'submission_knn_item.csv',
    'PMF':                  'submission_pmf.csv',
    'NMF':                  'submission_nmf.csv',
    modelo_bnmf:            'submission_bnmf.csv',
}
df_res['submission'] = df_res['modelo'].map(nombre_to_csv).fillna('—')

print('\n' + '═'*60)
print('  RANKING FINAL — VAL MAE (lower is better)')
print('═'*60)
display(df_res)

# ── Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_res))]
bars = ax.barh(df_res['modelo'][::-1], df_res['MAE'][::-1],
               color=colors[::-1], edgecolor='white')
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
ax.set_xlabel('MAE (val)')
ax.set_title('Comparativa de algoritmos — Recomendación')
ax.axvline(df_res['MAE'].iloc[0], color='green', linestyle='--', alpha=0.5, label='Mejor')
ax.legend()
plt.tight_layout()
plt.savefig('comparativa_algoritmos_submissions.png', bbox_inches='tight')
plt.show()

print('\n✅ Submissions generadas:')
for modelo, csv in nombre_to_csv.items():
    existe = '✓' if os.path.exists(csv) else '✗'
    print(f'  {existe}  {csv}')